# Task 2 of 3 (`Process_Medallion`)
### Lakeflow Jobs Orchestration Lab · CDC + Medallion Architecture
**Databricks Free Edition (serverless)**

This is the **second task** in the Job. It is the core of the lab, where we apply the
**Medallion architecture to a CDC feed**.

Here we use a **Notebook task** instead of a Pipeline task for two specific Free Edition limitations:

1. **`AUTO CDC INTO`** (the declarative command that replaced `APPLY CHANGES INTO`)
   **can only run inside a** Lakeflow Declarative Pipelines **pipeline**, not within a notebook cell.
2. Free Edition allows **only one active pipeline per pipeline type**, making it fragile to
   depend on a pipeline inside the Job for a lab environment.

For this reason, we manually reproduce the same logic performed by
`AUTO CDC INTO` using **`MERGE INTO`**. At the end, we also show the equivalent official syntax.

**Depends on:** `Land_New_Data` (Task 1).

In [0]:
# Shared configuration (identical across the 3 Job tasks)
catalog = "workspace"
schema  = "medallion_dbsql"
volume  = "raw_data"

base_path = f"/Volumes/{catalog}/{schema}/{volume}"
landing   = f"{base_path}/landing"     # CDC feed events are landed here

# Ensure the required structure exists (idempotent)
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {catalog}.{schema}.{volume}")
spark.sql(f"USE {catalog}.{schema}")

print("catalog/schema :", f"{catalog}.{schema}")
print("landing        :", landing)

catalog/schema : workspace.medallion_dbsql
landing        : /Volumes/workspace/medallion_dbsql/raw_data/landing


## Bronze Layer (Landing the Feed As-Is)

The Bronze layer is a **faithful copy** of the CDC feed. We store *every change event* exactly as it arrives,
including the `operation` and `sequence_num` columns. **Nothing is collapsed or transformed.**

We rebuild the Bronze table by reading **the entire landing directory** on every run, ensuring that the complete
history of the feed remains available and reproducible within the Job.

In [0]:
# BRONZE: Ingest ALL events from the landing directory without transformations
from pyspark.sql.functions import current_timestamp, col

raw_df = (
    spark.read
      .option("multiLine", "false")     # JSON Lines: one JSON object per line
      .json(landing)
)

bronze_df = (
    raw_df
      .withColumn("_ingested_at", current_timestamp())
      .withColumn("_source_file", col("_metadata.file_path"))
)

# Rebuild Bronze from the entire available feed (idempotent within the Job)
(bronze_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_orders_cdc"))

print("Bronze rebuilt from the entire landing directory.")
spark.sql("""
          SELECT order_id, country, category, status, operation, sequence_num 
          FROM bronze_orders_cdc 
          ORDER BY sequence_num
          """).show()

Bronze rebuilt from the entire landing directory.
+--------+--------------+-----------+-------+---------+------------+
|order_id|       country|   category| status|operation|sequence_num|
+--------+--------------+-----------+-------+---------+------------+
|       1|      Colombia|   Footwear|pending|   INSERT|           1|
|       2|        Mexico|   Clothing|pending|   INSERT|           2|
|       3|     Argentina|Accessories|pending|   INSERT|           3|
|       4|         Chile|   Clothing|pending|   INSERT|           4|
|       5|          Peru|  Outerwear|pending|   INSERT|           5|
|       6|       Ecuador|       Bags|pending|   INSERT|           6|
|       7|         Spain|Accessories|pending|   INSERT|           7|
|       8| United States|Accessories|pending|   INSERT|           8|
|       9|        Brazil|   Clothing|pending|   INSERT|           9|
|      10|       Uruguay|   Footwear|pending|   INSERT|          10|
|      11|    Costa Rica|Accessories|pending|   INSER

## Silver Layer (Applying the Changes) — "The Real CDC"

The Silver layer represents the **current state** of the data: **one row per `order_id`** containing its latest version.

The CDC pattern consists of two steps:

1. **Deduplicate the feed** by keeping **only the latest event for each `order_id`** based on
   `sequence_num` (a `MERGE` statement cannot process multiple rows with the same key simultaneously).
2. **`MERGE INTO`**: perform an upsert for all non-`DELETE` operations and remove rows when the operation is `DELETE`.

In [0]:
%sql
-- Silver table: the current state (without CDC feed metadata)
CREATE OR REPLACE TABLE silver_orders (
  order_id  INT,
  customer  STRING,
  country   STRING,
  city      STRING,
  category  STRING,
  product   STRING,
  quantity  INT,
  unit_price DOUBLE,
  amount    DOUBLE,
  status    STRING,
  order_date DATE
)

In [0]:
# SILVER: Apply the CDC feed using window-based deduplication + MERGE INTO
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, to_date

bronze = spark.table("bronze_orders_cdc")

# 1) Keep the latest event per key (order_id) based on the sequence number
w = Window.partitionBy("order_id").orderBy(col("sequence_num").desc())
latest = (
    bronze
      .withColumn("_rn", row_number().over(w))
      .filter(col("_rn") == 1)          # Keep only the most recent change per order
      .drop("_rn")
      .withColumn("order_date", to_date(col("order_date")))
      .withColumn("quantity", col("quantity").cast("int"))
)
latest.createOrReplaceTempView("latest_cdc")

# 2) MERGE: Apply the latest state to Silver
spark.sql("""
    MERGE INTO silver_orders AS t
    USING latest_cdc         AS s
      ON t.order_id = s.order_id
    WHEN MATCHED AND s.operation = 'DELETE' THEN DELETE
    WHEN MATCHED AND s.operation != 'DELETE' THEN UPDATE SET
        t.customer = s.customer, t.country = s.country,
        t.city = s.city, t.category = s.category,
        t.product = s.product, t.quantity = s.quantity,
        t.unit_price = s.unit_price, t.amount   = s.amount,
        t.status  = s.status, t.order_date = s.order_date
    WHEN NOT MATCHED AND s.operation != 'DELETE' THEN INSERT
        (order_id, customer, country, city, category, product, quantity, unit_price, amount, status, order_date)
        VALUES (s.order_id, s.customer, s.country, s.city, s.category,s.product, s.quantity, s.unit_price , s.amount, s.status, s.order_date)
""")

print("CDC applied to silver_orders.")

CDC applied to silver_orders.


In [0]:
%sql
-- Silver: the CURRENT STATE after applying the CDC changes
SELECT *
FROM silver_orders
ORDER BY order_id

order_id,customer,country,city,category,product,quantity,unit_price,amount,status,order_date
1,Ana,Colombia,Bogota,Footwear,Sneakers,1,90.0,90.0,pending,2026-07-01
2,Beto,Mexico,Mexico City,Clothing,T-Shirt,2,25.0,50.0,pending,2026-07-01
3,Carla,Argentina,Buenos Aires,Accessories,Cap,1,15.0,15.0,pending,2026-07-02
4,Diego,Chile,Santiago,Clothing,Jeans,1,65.0,65.0,pending,2026-07-02
5,Elena,Peru,Lima,Outerwear,Jacket,1,120.0,120.0,pending,2026-07-03
6,Fabio,Ecuador,Quito,Bags,Backpack,1,55.0,55.0,pending,2026-07-03
7,Gabriela,Spain,Madrid,Accessories,Watch,1,210.0,210.0,pending,2026-07-04
8,Hugo,United States,Miami,Accessories,Sunglasses,1,80.0,80.0,pending,2026-07-04
9,Isabel,Brazil,Sao Paulo,Clothing,Sweater,2,45.0,90.0,pending,2026-07-05
10,Javier,Uruguay,Montevideo,Footwear,Boots,1,140.0,140.0,pending,2026-07-05


## Gold Layer (Two business datasets)

Gold adds the current status (Silver) to **business metrics**. Since it builds upon the already established Silver, any order deleted by a `DELETE` command **doesn't count** (exactly what we want).

We create **two Gold tables**, which will be the **datasets for the dashboard:**
1. `gold_sales_region`: Metrics by country (orders, units, amount, average ticket)

2. `gold_sales_category_day`: Series by date x category x country (for tracking trends over time and to ensure the country filter applies to both datasets).

In [0]:
%sql
-- GOLD: Metrics by region/country
CREATE OR REPLACE TABLE gold_sales_region AS
SELECT
  country                           AS country,
  COUNT(*)                          AS num_orders,
  SUM(quantity)                      AS units,
  ROUND(SUM(amount), 2)             AS total_amount,
  ROUND(SUM(amount) / COUNT(*), 2)  AS average_ticket
FROM silver_orders
GROUP BY country;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- GOLD: Time series by category (and country, for filtering)
CREATE OR REPLACE TABLE gold_sales_category_day AS
SELECT
  order_date,
  category,
  country,
  COUNT(*)                          AS num_orders,
  SUM(quantity)                     AS units,
  ROUND(SUM(amount), 2)             AS total_amount
FROM silver_orders
GROUP BY order_date, category, country;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM gold_sales_region
ORDER BY total_amount DESC;

country,num_orders,units,total_amount,average_ticket
Spain,1,1,210.0,210.0
Australia,1,1,180.0,180.0
Uruguay,1,1,140.0,140.0
Peru,1,1,120.0,120.0
United Kingdom,1,1,110.0,110.0
Canada,1,1,95.0,95.0
Colombia,1,1,90.0,90.0
Brazil,1,2,90.0,90.0
United States,1,1,80.0,80.0
Chile,1,1,65.0,65.0


In [0]:
%sql
SELECT *
FROM gold_sales_category_day
ORDER BY order_date, category

order_date,category,country,num_orders,units,total_amount
2026-07-01,Clothing,Mexico,1,2,50.0
2026-07-01,Footwear,Colombia,1,1,90.0
2026-07-02,Accessories,Argentina,1,1,15.0
2026-07-02,Clothing,Chile,1,1,65.0
2026-07-03,Bags,Ecuador,1,1,55.0
2026-07-03,Outerwear,Peru,1,1,120.0
2026-07-04,Accessories,Spain,1,1,210.0
2026-07-04,Accessories,United States,1,1,80.0
2026-07-05,Clothing,Brazil,1,2,90.0
2026-07-05,Footwear,Uruguay,1,1,140.0


## The Production Version: `AUTO CDC INTO` (Reference Only)

Everything we implemented manually using `MERGE INTO` and window-based deduplication is **exactly what Databricks automates** with `AUTO CDC INTO` (formerly `APPLY CHANGES INTO`). However, this functionality must run **inside a Lakeflow Declarative Pipelines pipeline**, not within a notebook task.

```sql
-- (This belongs INSIDE a Lakeflow Declarative Pipelines pipeline)
CREATE OR REFRESH STREAMING TABLE silver_orders;

CREATE FLOW silver_orders_flow AS AUTO CDC INTO silver_orders
FROM STREAM(bronze_orders_cdc)
KEYS (order_id)
APPLY AS DELETE WHEN operation = "DELETE"
SEQUENCE BY sequence_num
COLUMNS * EXCEPT (operation, sequence_num);
```

| Our Manual Implementation | `AUTO CDC INTO` Clause |
|---|---|
| `MERGE ... ON t.order_id = s.order_id` | `KEYS (order_id)` |
| `WHEN MATCHED AND operation='DELETE' THEN DELETE` | `APPLY AS DELETE WHEN operation="DELETE"` |
| `Window.partitionBy(...).orderBy(sequence_num.desc())` + `row_number` | `SEQUENCE BY sequence_num` |
| Upsert (`INSERT` / `UPDATE SET`) | Default behavior |
| `.drop("operation", "sequence_num")` | `COLUMNS * EXCEPT (operation, sequence_num)` |

---

**Task 2 completed.** The Bronze, Silver, and Gold layers have been updated.

**Task 3** (`Verify_Output`) will read the Gold layer to confirm the final results.